## OpenAlex API Queries


### Imports and global variables

In [1]:
import csv
import sys
import time
import requests
import os

API_KEY = "8Tb0QywKVVtbUVWmxwY4dp"          # insert your API key here or set OPENALEX_API_KEY env var
MAILTO = "ui@openalex.org"  
PER_PAGE = 200   

### API call 1: terms in the abstract or title, but not in the main text.

This API call returns which don’t have either of the terms (sonification and audification) in the abstract and title but do have at least one of the terms in the main text – what do these papers look like? 

Notes: 

- full-text search only has a sub-set of OpenAlex entries; so the dataset generated is not complete.
- not all OpenAlex entries have abstracts
- it is possible OpenAlex's search index (title_abstract search) and stored fields (from inverted index) disagree.
- there isn't a main text search, so we do a full text search, and then the title and abstract search.
- we subtract the title and abstract search dataset from the full text search dataset
- each term (sonification / audification) is run as its own query then the results are merged (removing duplicates).

### Variables

In [2]:
TERMS = ["sonification", "audification"]
TYPE_FILTER = "article|preprint|book-chapter|dissertation|book|review|letter|report|editorial"

BASE = "https://api.openalex.org/works"  
OUTPUT_CSV = "../data/openAlex/fulltext_only.csv"

# Fields to retrieve (keeps download small). Remove `select` to get everything.
SELECT = "id,doi,title,display_name,publication_year,type,primary_location,abstract_inverted_index"

### Functions

Generated with Claude.

In [3]:
def paginate(filter_str: str, session: requests.Session, api_key: str) -> dict:
    """Cursor-paginate through every page for a filter string. Returns {id: record}."""
    out = {}
    cursor = "*"
    page = 0
 
    while cursor:
        params = {
            "filter": filter_str,
            "per-page": PER_PAGE,
            "cursor": cursor,
            "select": SELECT,
            "mailto": MAILTO,
        }
        if api_key:
            params["api_key"] = api_key
 
        for attempt in range(5):
            resp = session.get(BASE, params=params, timeout=60)
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 30))
                print(f"    [429] backing off {wait}s ...", file=sys.stderr)
                time.sleep(wait)
                continue
            if resp.status_code == 400:
                print(f"    [400] {resp.text}", file=sys.stderr)
            resp.raise_for_status()
            break
        else:
            raise RuntimeError(f"Repeatedly rate-limited on filter: {filter_str}")
 
        data = resp.json()
        batch = data.get("results", [])
        for w in batch:
            out[w["id"]] = w
 
        page += 1
        total = data["meta"].get("count")
        print(f"    page {page}: +{len(batch)} (running {len(out)}/{total})")
 
        cursor = data["meta"].get("next_cursor")
        if not batch:
            break
        time.sleep(0.1)
 
    return out

def reconstruct_abstract(inv) -> str:
    """Rebuild abstract text from OpenAlex's {word: [positions]} inverted index."""
    if not inv:
        return ""
    pairs = []
    for word, positions in inv.items():
        for p in positions:
            pairs.append((p, word))
    pairs.sort(key=lambda x: x[0])
    return " ".join(w for _, w in pairs)
 
def title_or_abstract_text(work: dict) -> str:
    # Check title AND display_name: OpenAlex's searchable index can disagree with
    # the stored record, so we look at every field that could hold the term.
    title = work.get("title") or ""
    display = work.get("display_name") or ""
    abstract = reconstruct_abstract(work.get("abstract_inverted_index"))
    return f"{title} {display} {abstract}".lower()

In [4]:
session = requests.Session()
terms_lower = [t.lower() for t in TERMS]

# fulltext.search 
candidates = {}
fulltext_by_term = {}
for term in TERMS:
    print(f"fulltext.search:{term}")
    hits = paginate(f"fulltext.search:{term},type:{TYPE_FILTER}", session, API_KEY)
    fulltext_by_term[term] = set(hits)
    candidates.update(hits)
    print(f"  -> {len(hits)} works mention '{term}' somewhere\n")
print(f"Candidate pool (term anywhere, deduplicated): {len(candidates)}")

# API title/abstract search
ta_exclude = set()
for term in TERMS:
    print(f"title_and_abstract.search:{term}")
    ta_hits = paginate(f"title_and_abstract.search:{term},type:{TYPE_FILTER}", session, API_KEY)
    ta_exclude |= set(ta_hits)
    print(f"  -> {len(ta_hits)} works with '{term}' in title/abstract\n")

# stored title abstract search
kept = {}
dropped_local = 0
dropped_api = 0
for wid, w in candidates.items():
    text = title_or_abstract_text(w)
    if any(t in text for t in terms_lower):
        dropped_local += 1
        continue
    if wid in ta_exclude:          # API caught it but the literal text didn't
        dropped_api += 1
        continue
    kept[wid] = w
print(f"Dropped (literal term in title/abstract): {dropped_local}")
print(f"Dropped (API title/abstract search only): {dropped_api}")
print(f"Body-only works: {len(kept)}\n")

fulltext.search:sonification
    page 1: +200 (running 200/15945)
    page 2: +200 (running 400/15945)
    page 3: +200 (running 600/15945)
    page 4: +200 (running 800/15945)
    page 5: +200 (running 1000/15945)
    page 6: +200 (running 1200/15945)
    page 7: +200 (running 1400/15945)
    page 8: +200 (running 1600/15945)
    page 9: +200 (running 1800/15945)
    page 10: +200 (running 1999/15945)
    page 11: +200 (running 2199/15945)
    page 12: +200 (running 2398/15945)
    page 13: +200 (running 2598/15945)
    page 14: +200 (running 2798/15945)
    page 15: +200 (running 2996/15945)
    page 16: +200 (running 3196/15945)
    page 17: +200 (running 3395/15945)
    page 18: +200 (running 3593/15945)
    page 19: +200 (running 3793/15945)
    page 20: +200 (running 3993/15945)
    page 21: +200 (running 4192/15945)
    page 22: +200 (running 4391/15945)
    page 23: +200 (running 4591/15945)
    page 24: +200 (running 4791/15945)
    page 25: +200 (running 4991/15945)
    page 

In [5]:
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["openalex_id", "doi", "title", "year", "type",
                     "source", "matched_terms", "abstract_available"])
    for wid, w in kept.items():
        loc = w.get("primary_location") or {}
        src = (loc.get("source") or {}).get("display_name", "")
        abstract_avail = "yes" if w.get("abstract_inverted_index") else "no"
        matched = "|".join(t for t in TERMS if wid in fulltext_by_term[t])
        writer.writerow([
            wid,
            w.get("doi", ""),
            w.get("title") or w.get("display_name", ""),
            w.get("publication_year", ""),
            w.get("type", ""),
            src,
            matched,
            abstract_avail,
        ])
print(f"Wrote {len(kept)} rows to {OUTPUT_CSV}")

Wrote 10935 rows to ../data/openAlex/fulltext_only.csv


In [8]:
TA_QUERY = "sonification AND audification"   # both terms in title/abstract

print(f"title_and_abstract.search:{TA_QUERY}")
ta_works = paginate(
    f"title_and_abstract.search:{TA_QUERY},type:{TYPE_FILTER}",
    session, API_KEY,
)
print(f"  -> {len(ta_works)} works with the terms in title/abstract\n")

title_and_abstract.search:sonification AND audification
    page 1: +60 (running 60/60)
  -> 60 works with the terms in title/abstract



In [9]:
4992+112-60    # result on website is 5049. So **pretty much** consistent (though why it differs at all is still weird).

5044